In [1]:
from pathlib import Path
import json
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
from sqlalchemy import text

PROJECT_DIR = Path.cwd().parent

print(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))
from database import get_engine

/home/sgurung/Desktop/final_nyc_citi_bike_forecast


In [ ]:
import os
import getpass

os.environ["PGUSER"] = "sgurung"
os.environ["PGDATABASE"] = "citibike"
os.environ["PGPASSWORD"] = getpass.getpass("PostgreSQL password: ")

In [3]:
FORECAST_TABLE = "hourly_neighborhood_forecast"
NEIGHBORHOOD_URL = "https://data.cityofnewyork.us/resource/9nt8-h7nd.geojson?$limit=500"

In [4]:
engine = get_engine()
with engine.connect() as connection:
    forecast = pd.read_sql(text(
        f"""Select 
        hour, nta_code, nta_name, borough, 
        predicted_arrivals, predicted_departures,predicted_net_flow
        From {FORECAST_TABLE}"""
    ), con=connection)
forecast.head()

,hour,nta_code,nta_name,borough,predicted_arrivals,predicted_departures,predicted_net_flow
0,2026-09-07 17:00:00,BK0101,Greenpoint,Brooklyn,399.918800,341.023900,58.894897
1,2026-09-07 17:00:00,BK0102,Williamsburg,Brooklyn,738.527340,696.534500,41.992860
2,2026-09-07 17:00:00,BK0103,South Williamsburg,Brooklyn,33.965153,33.812115,0.153038
3,2026-09-07 17:00:00,BK0104,East Williamsburg,Brooklyn,285.863600,261.182740,24.680847
4,2026-09-07 17:00:00,BK0201,Brooklyn Heights,Brooklyn,188.415450,150.337280,38.078170


In [5]:
forecast.isna().sum()

hour                    0
nta_code                0
nta_name                0
borough                 0
predicted_arrivals      0
predicted_departures    0
predicted_net_flow      0
dtype: int64

In [6]:
forecast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21000 entries, 0 to 20999
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   hour                  21000 non-null  datetime64[ns]
 1   nta_code              21000 non-null  object        
 2   nta_name              21000 non-null  object        
 3   borough               21000 non-null  object        
 4   predicted_arrivals    21000 non-null  float64       
 5   predicted_departures  21000 non-null  float64       
 6   predicted_net_flow    21000 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(3)
memory usage: 1.1+ MB


In [7]:
# Visuals

In [8]:
boundaries = gpd.read_file(NEIGHBORHOOD_URL)
boundaries = boundaries.rename(columns={"nta2020": "nta_code"})
boundaries = boundaries[["nta_code", "geometry"]].copy()
boundaries["nta_code"] = boundaries["nta_code"].astype(str)
boundaries = boundaries.to_crs("EPSG:4326")

In [9]:
selected_hour = forecast["hour"].min()
selected = forecast.loc[forecast["hour"] == selected_hour].copy()

print(f"Selected hour: {selected_hour}")
print(f"Neighborhoods shown: {len(selected):,}")

Selected hour: 2026-09-07 17:00:00
Neighborhoods shown: 125


In [ ]:
map_data = boundaries.merge(
    selected,
    on="nta_code",
    how="inner",
    validate="one_to_one",
)
geojson = json.loads(map_data[["nta_code", "geometry"]].to_json())
plot_data = pd.DataFrame(map_data.drop(columns="geometry"))
largest_flow = max(float(plot_data["predicted_net_flow"].abs().max()), 1.0)

net_flow_map = px.choropleth_map(
    plot_data,
    geojson=geojson,
    locations="nta_code",
    featureidkey="properties.nta_code",
    color="predicted_net_flow",
    color_continuous_scale=["#FF2600", "#FFFFFF", "#84FF00"],
    range_color=(-largest_flow, largest_flow),
    hover_name="nta_name",
    hover_data={
        "nta_code": False,
        "borough": True,
        "predicted_arrivals": ":,.1f",
        "predicted_departures": ":,.1f",
        "predicted_net_flow": ":+,.1f",
    },
    center={"lat": 40.71, "lon": -74},
    zoom=9.5,
    map_style="carto-positron",
    title=f"Predicted neighborhood net flow: {selected_hour:%b %d, %Y %I:%M %p}"
)
net_flow_map.update_traces(marker_line_width=0.5, marker_line_color="#000208")
net_flow_map.update_layout(height=650, margin={"l": 0, "r": 0, "t": 50, "b": 0})
net_flow_map.update_geos(fitbounds="locations", visible=False)
net_flow_map.show()

In [ ]:
hourly_flow = forecast.groupby("hour", as_index=False)[["predicted_arrivals", "predicted_departures"]].sum().rename(columns={
    "predicted_arrivals": "Arrivals",
    "predicted_departures": "Departures"})

trend_data = hourly_flow.melt(
    id_vars="hour",
    value_vars=["Arrivals", "Departures"],
    var_name="Flow type",
    value_name="Predicted rides"
)

flow_trend = px.line(
    trend_data,
    x="hour",
    y="Predicted rides",
    color="Flow type",
    color_discrete_map={
        "Arrivals": "#83FD00",
        "Departures": "#FF2600"
    },
    title="Hourly predicted arrivals and departures"
)
flow_trend.update_layout(
    height=420,
    xaxis_title=None,
    yaxis_title="Predicted rides",
    legend_title_text=None,
    hovermode="x unified"
)
flow_trend.update_yaxes(rangemode="tozero")
flow_trend.show()

In [77]:
shortages = selected.nsmallest(5, "predicted_net_flow")
surpluses = selected.nlargest(5, "predicted_net_flow")
ranking = pd.concat([shortages, surpluses]).drop_duplicates("nta_code")
ranking = ranking.sort_values("predicted_net_flow")
ranking["Direction"] = np.where(
    ranking["predicted_net_flow"] < 0,
    "Shortage",
    "Surplus"
)

imbalance_chart = px.bar(
    ranking,
    x="predicted_net_flow",
    y="nta_name",
    orientation="h",
    color="Direction",
    color_discrete_map={
        "Shortage": "#FF2600",
        "Surplus": "#83FD00"
    },
    text_auto=":,.1f",
    labels={
        "predicted_net_flow": "Predicted net flow",
        "nta_name": "Neighborhood"
    },
    title=f"Largest neighborhood imbalances: {selected_hour:%b %d, %I:%M %p}"
)
imbalance_chart.add_vline(x=0, line_color="#000000", line_width=0.5)
imbalance_chart.update_layout(height=450, yaxis_title=None, legend_title_text=None)
imbalance_chart.show()